# How to Use 3 Types of Trendline in Plotly Express

## Plotly Express supports OLS, Rolling Mean and LOWESS trendlines; we use Summer Temperature Anomalies (1925–2025) from NASA GISS to illustrate their use


When you plot real-world data over time — global temperatures, polling averages, stock prices, anything measured repeatedly — you may find that the underlying trend you care about is buried under noise. We are going to describe three tools for pulling that trend out of the noise; although they answer subtly different questions.

Those tools are **OLS**, **Rolling Mean**, and **LOWESS**.

This code and related data support a tutorial published on Medium and Substack. Finds links on my [web site](https://alanjones2.github.io/)



In [19]:
import pandas as pd
import plotly.express as px

px.defaults.template = "plotly_dark"
px.defaults.height = 500
px.defaults.width = 1000



## 1. Load the data

In [20]:
df = pd.read_csv("data/NH.Ts+dSST-1925-2025.csv")
#df.head()
summer = df[["Year", "Jun", "Jul", "Aug", "JJA"]].copy()
print(f"Years: {summer['Year'].min()}–{summer['Year'].max()} ({len(summer)} records)")
summer.head(10)

Years: 1925–2025 (101 records)


,Year,Jun,Jul,Aug,JJA
0,1925,-0.26,-0.26,-0.21,-0.24
1,1926,-0.21,-0.20,-0.07,-0.16
2,1927,-0.15,-0.09,-0.16,-0.13
3,1928,-0.33,-0.11,-0.19,-0.21
4,1929,-0.40,-0.36,-0.21,-0.32
5,1930,-0.12,-0.04,0.01,-0.05
6,1931,0.15,0.07,0.09,0.10
7,1932,-0.15,-0.15,-0.08,-0.12
8,1933,-0.32,-0.14,-0.15,-0.20
9,1934,-0.02,-0.05,-0.11,-0.06


In [21]:
summer.describe()

,Year,Jun,Jul,Aug,JJA
count,101.000000,101.000000,101.000000,101.000000,101.000000
mean,1975.000000,0.264554,0.257822,0.269109,0.264455
std,29.300171,0.431737,0.414272,0.427554,0.420729
min,1925.000000,-0.400000,-0.360000,-0.280000,-0.320000
25%,1950.000000,-0.050000,-0.040000,-0.050000,-0.030000
50%,1975.000000,0.140000,0.090000,0.100000,0.110000
75%,2000.000000,0.500000,0.500000,0.550000,0.500000
max,2025.000000,1.540000,1.450000,1.570000,1.460000


## 2. Summer (JJA) anomalies + trendlines

This section plots each summer's JJA anomaly and overlays trendlines. All trendlines below
are generated by Plotly Express via `px.scatter(..., trendline=...)` — no manual fits.

In [22]:
fig = px.scatter(
    summer,
    x="Year",
    y="JJA",
    title="Summer (JJA) Global Temperature Anomalies, 1925–2025",
    labels={"JJA": "Anomaly (°C vs 1951–1980)"},
)
#fig.update_traces(marker=dict(color="#9ca3af", size=6, opacity=0.8), selector=dict(mode="markers"))
fig.add_hline(y=0, line_dash="dot", line_color="white", opacity=0.6,
              annotation_text="1951–1980 baseline", annotation_position="bottom right")
fig.update_layout(height=500, hovermode="closest")
fig.show()

In [23]:
fig = px.line(
    summer,
    x="Year",
    y="JJA",
    title="Summer (JJA) Global Temperature Anomalies, 1925–2025",
    labels={"JJA": "Anomaly (°C vs 1951–1980)"},
    hover_data={"Year": True, "JJA": ":+.2f"},
)
fig.update_traces(marker=dict(color="#9ca3af", size=6, opacity=0.8), selector=dict(mode="markers"))
fig.add_hline(y=0, line_dash="dot", line_color="white", opacity=0.6,
              annotation_text="1951–1980 baseline", annotation_position="bottom right")
fig.show()

In [24]:
figols = px.scatter(
    summer,
    x="Year",
    y="JJA",
    trendline="ols",
    trendline_color_override="#1f3b57",
    title="Summer (JJA) Global Temperature Anomalies + OLS Trend, 1925–2025",
    labels={"JJA": "Anomaly (°C vs 1951–1980)"},
    #hover_data={"Year": True, "JJA": ":+.2f"},
)
#figols.update_traces(marker=dict(color="#9ca3af", size=6, opacity=0.8), selector=dict(mode="markers"))
figols.add_hline(y=0, line_dash="dot", line_color="white", opacity=0.6,
              annotation_text="1951–1980 baseline", annotation_position="bottom right")
figols.show()

### OLS fit statistics

## 3. 10-year rolling-mean trend

In [25]:
fig_roll = px.scatter(
    summer,
    x="Year",
    y="JJA",
    trendline="rolling",
    trendline_options=dict(window=10),
    trendline_color_override="#ef553b",
    title="Summer Anomalies with 10-Year Rolling-Mean Trend",
    labels={"JJA": "Anomaly (°C)"},
)
#fig_roll.update_traces(marker=dict(color="#9ca3af", size=5, opacity=0.7), selector=dict(mode="markers"))
fig_roll.add_hline(y=0, line_dash="dot", line_color="white", opacity=0.6)
fig_roll.show()

## 4. LOWESS smooth trend

In [26]:
fig_lowess = px.scatter(
    summer,
    x="Year",
    y="JJA",
    trendline="lowess",
    trendline_options=dict(frac=0.3),
    trendline_color_override="#00a08a",
    title="Summer Anomalies with LOWESS Smooth Trend",
    labels={"JJA": "Anomaly (°C)"},
)
#fig_lowess.update_traces(marker=dict(color="#9ca3af", size=5, opacity=0.7), selector=dict(mode="markers"))
fig_lowess.add_hline(y=0, line_dash="dot", line_color="white", opacity=0.6)
fig_lowess.show()

## 5. June vs July vs August — one OLS trendline per month

When `color=` is set, Plotly Express fits a separate `ols` trendline per group automatically.

In [27]:
monthly = summer.melt(id_vars="Year", value_vars=["Jun", "Jul", "Aug"],
                      var_name="Month", value_name="Anomaly")
fig3 = px.scatter(
    monthly,
    x="Year", y="Anomaly", color="Month",
    trendline="ols",
    title="Summer Months with Per-Month OLS Trendlines",
    labels={"Anomaly": "Anomaly (°C)"},
)
fig3.add_hline(y=0, line_dash="dot", line_color="white", opacity=0.5)
fig3.show()

In [28]:
monthly = summer.melt(id_vars="Year", value_vars=["Jun", "Jul", "Aug"],
                      var_name="Month", value_name="Anomaly")
fig3 = px.scatter(
    monthly,
    x="Year", y="Anomaly", color="Month",
    trendline="lowess",
    trendline_options=dict(frac=0.3),
    title="Summer Months with Per-Month LOWESS",
    labels={"Anomaly": "Anomaly (°C)"},
)
fig3.add_hline(y=0, line_dash="dot", line_color="white", opacity=0.5)
fig3.show()

## 6. June vs July vs August — separate monthly points, single overall trendline

Chart 6 uses `color="Month"` + `trendline="ols"`, so px fits one trendline **per month**.
Here we keep the per-month coloured points but show only **one pooled OLS trendline**.
Both layers are built with Plotly Express — the trendline itself is still generated by
`px.scatter(..., trendline="ols")` (no `color=` → single fit), then overlaid.


In [29]:
fig3_single = px.scatter(
    monthly,
    x="Year", y="Anomaly", color="Month",
    title="Summer Months: Separate Points, Single Overall OLS Trendline",
    labels={"Anomaly": "Anomaly (°C)"},
        trendline="ols",
    trendline_color_override="white",
    trendline_scope="overall"
)

fig3_single.add_hline(y=0, line_dash="dot", line_color="white", opacity=0.5)
fig3_single.show()

In [30]:
# 6b: coloured monthly points (no per-group trendlines here)
fig3_single = px.scatter(
    monthly,
    x="Year", y="Anomaly", color="Month",
    title="Summer Months: Separate Points, Single Overall LOWESS",
    labels={"Anomaly": "Anomaly (°C)"},        
    trendline="lowess",
    trendline_color_override="white",
    trendline_scope="overall"
)

fig3_single.add_hline(y=0, line_dash="dot", line_color="white", opacity=0.5)
fig3_single.show()

